# Filtered summary data from ag_ab dataframe
--> to only download pdb files we actually need and want and not all of them

In [26]:
import pandas as pd
import os.path

# path to dataframe
df = pd.read_csv("../data/ab_ag.tsv", sep="\t")

df_filtered= (df
                .assign(resolution = pd.to_numeric(df["resolution"], errors='coerce'))
                .get(["pdb", "Hchain", "Lchain", "model", "antigen_chain",
                        "antigen_type", "antigen_name", "compound", "organism",
                        "heavy_species", "light_species", "antigen_species",
                        "resolution", "method", "scfv", "engineered", 
                        "heavy_subclass", "light_subclass", "light_ctype"])
                .dropna(subset=["pdb", "Hchain", "Lchain", 
                                "model", "antigen_chain", "antigen_type",
                                "antigen_species", "method"])
                .query("scfv== False and resolution <= 3.25 and antigen_type == 'protein'")
                .query("pdb not in['6erx', '7ctu', '3sm5', '3l5y']")
                .groupby("pdb")
                .head(1))

df_filtered.shape
                

(1485, 19)

In [27]:
df_filtered["species"] = ""
df_filtered.loc[df_filtered["antigen_species"].str.contains("coronavirus2", case = False, na = False), "species"] = "SARS-CoV-2"
df_filtered.loc[df_filtered["antigen_species"].str.contains("homo sapiens", case = False, na = False), "species"] = "Homo Sapiens"
df_filtered.loc[df_filtered["antigen_species"].str.contains("influenza a", case = False, na = False), "species"] = "Influenza A"
df_filtered = df_filtered[df_filtered["species"] != ""]

df_filtered.species.value_counts()

df_filtered.shape




(962, 20)

In [28]:
df_filtered.antigen_chain.unique()

array(['E', 'A', 'D', 'B', 'C', 'R', 'H', 'I', 'P', 'F', 'X', 'K', 'Z',
       'J', 'Q', 'T', 'G', 'M', 'V', 'S', 'c', 'Y', 'U', 'm', 'b', 'N',
       'O', 'W'], dtype=object)

In [29]:
os.makedirs("../generated/data cleanup", exist_ok=True)
df_filtered.to_csv("../generated/data cleanup/ab_ag_filtered_pdb.tsv", sep="\t", index = False)



In [30]:
df_filtered

,pdb,Hchain,Lchain,model,antigen_chain,antigen_type,antigen_name,compound,organism,heavy_species,light_species,antigen_species,resolution,method,scfv,engineered,heavy_subclass,light_subclass,light_ctype,species
5,8veb,G,I,0,E,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.97,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
8,8ved,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E11 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.98,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV2,Kappa,Influenza A
11,8vee,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,3.18,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
14,8vef,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 UCA (unm...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,3.04,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
20,9dpc,H,L,0,D,protein,neuraminidase,Structure of Fab 297 in complex with influenza...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.65,ELECTRON MICROSCOPY,False,True,IGHV1,IGKV1,Kappa,Influenza A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5516,7rp3,H,I,0,A,protein,isoform 2b of gtpase kras,Crystal structure of GNE-1952 alkylated KRAS G...,HOMO SAPIENS,homo sapiens,homo sapiens,homo sapiens,2.00,X-RAY DIFFRACTION,False,True,IGHV4,IGLV1,Lambda,Homo Sapiens
5517,3q1s,H,L,0,I,protein,interleukin-22,HIV-1 neutralizing antibody Z13e1 in complex w...,HOMO SAPIENS,homo sapiens,homo sapiens,homo sapiens,2.15,X-RAY DIFFRACTION,False,True,IGHV4,IGKV3D,Kappa,Homo Sapiens
5518,4zfg,H,L,0,A,protein,angiopoietin-2,Dual-specificity Fab 5A12 in complex with Angi...,HOMO SAPIENS,homo sapiens,homo sapiens,homo sapiens,2.27,X-RAY DIFFRACTION,False,True,IGHV3,IGKV1,Kappa,Homo Sapiens
5521,4ps4,H,L,0,A,protein,interleukin-13,Crystal structure of the complex between IL-13...,HOMO SAPIENS,homo sapiens,homo sapiens,homo sapiens,2.80,X-RAY DIFFRACTION,False,True,IGHV2,IGKV3D,Kappa,Homo Sapiens


## Pdb Liste mit eindeutigen pdbs für Sars CoV 2 (Corona)

In [31]:
# Alle eindeutigen PDB-IDs aus der gefilterten DataFrame holen
pdb_ids_sars_cov2 = df_filtered.query("species == 'SARS-CoV-2'").get(['pdb'])

#save to file
pdb_ids_sars_cov2.to_csv("pdb_ids_sars_cov2.csv", index=False, header=True)

pdb_ids_sars_cov2

,pdb
46,9cci
47,9ccj
52,9bj2
53,9bj3
108,8z6r
...,...
5470,7chp
5482,7n4i
5511,7l7d
5514,7n3d


## Pdb Liste mit eindeutigen pdbs für Homo Sapiens

In [32]:
# Alle eindeutigen PDB-IDs aus der gefilterten DataFrame holen
pdb_ids_homo_sapiens = df_filtered.query("species == 'Homo Sapiens'").get(['pdb'])

#save to file
pdb_ids_homo_sapiens.to_csv("pdb_ids_homo_sapiens.csv", index=False, header=True)

pdb_ids_homo_sapiens

,pdb
29,8rmx
31,8rmy
128,8znz
144,9avo
145,9awe
...,...
5515,4od2
5516,7rp3
5517,3q1s
5518,4zfg


## Pdb Liste mit eindeutigen pdbs für Influenza A

In [33]:
# Alle eindeutigen PDB-IDs aus der gefilterten DataFrame holen
pdb_ids_influenza = df_filtered.query("species == 'Influenza A'").get(['pdb'])

#save to file
pdb_ids_influenza.to_csv("pdb_ids_influenza.csv", index=False, header=True)

pdb_ids_influenza

,pdb
5,8veb
8,8ved
11,8vee
14,8vef
20,9dpc
...,...
5352,4kvn
5366,5vag
5391,4hfu
5396,6lxi
